# Cosmos Policy: uncertainty-aware planning in LIBERO

**Final experimental snapshot · 24 July 2026**

This notebook is the compact entry point for the completed standard LIBERO and
LIBERO-PRO experiments. It combines the confirmatory tables, plots,
interpretation and portable matched-seed videos. The authoritative detailed
report is [LIBERO_COMPLETE_RESULTS_20260724.md](LIBERO_COMPLETE_RESULTS_20260724.md).

Three evidence levels are kept separate:

1. **Confirmatory:** frozen calibration/holdout and pooled paired comparisons.
2. **Exploratory:** temporal detector search and small ablations.
3. **Visual case studies:** selected matched-seed pilot videos.

## 1. What was actually run

| Benchmark          | Role                           |   Executions | Success / fail   | Status                     |
|:-------------------|:-------------------------------|-------------:|:-----------------|:---------------------------|
| Standard LIBERO ID | pipeline control               |           72 | 72 / 0           | completed                  |
| LIBERO-PRO OOD     | boundary screening             |          106 | 82 / 24          | completed                  |
| LIBERO-PRO OOD     | detector + planning validation |         1078 | 603 / 475        | 23/23 jobs completed       |
| LIBERO-Safety      | official safety constraints    |            0 | -                | environment ready; not run |

The 1078 rows are **strategy executions**, not 1078 independent scenes:
identical `task/init_state/rollout_seed` combinations were repeated for
different planners. Planning conclusions therefore use paired seeds rather
than the pooled `603/1078` success fraction.

No newer experiment output was present on the MLSpace server at the time this
snapshot was built.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import HTML, Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "experiments":
    PROJECT_ROOT = PROJECT_ROOT.parent

FULL_DIR = (
    PROJECT_ROOT
    / "experiments/campaigns/libero_full_validation_20260724/analysis/full_validation"
)
summary = json.loads((FULL_DIR / "summary.json").read_text(encoding="utf-8"))
cases = pd.read_csv(FULL_DIR / "case_outcome_summary.csv")
planning = pd.read_csv(FULL_DIR / "planning_strategy_summary.csv")
pooled = pd.read_csv(FULL_DIR / "pooled_planning_confirmatory.csv")
detector = pd.read_csv(FULL_DIR / "prespecified_detector_exact_query.csv")
prediction_error = pd.read_csv(FULL_DIR / "prediction_error_correlations_q0_5.csv")

print(
    f"Loaded {summary['episodes']} strategy executions and "
    f"{summary['query_rows']} policy queries."
)

## 2. Experimental coverage

| Case                        | Split          |   Executions |   Success |   Fail | Success rate   |
|:----------------------------|:---------------|-------------:|----------:|-------:|:---------------|
| direct_milk_task5_init0     | calibration    |           36 |        17 |     19 | 47.2%          |
| direct_milk_task5_init0     | holdout        |           36 |        16 |     20 | 44.4%          |
| direct_milk_task5_new_inits | screen         |           30 |        30 |      0 | 100.0%         |
| direct_mug_task0_init0      | generalization |           24 |        19 |      5 | 79.2%          |
| long_mug_task4_init0        | generalization |          124 |       101 |     23 | 81.5%          |
| milk_task5_init0            | calibration    |          232 |       129 |    103 | 55.6%          |
| milk_task5_init0            | holdout        |          232 |       110 |    122 | 47.4%          |
| milk_task5_init0_ablation   | generalization |          146 |        60 |     86 | 41.1%          |
| yellow_task8_init0          | generalization |          194 |        98 |     96 | 50.5%          |
| yellow_task8_new_inits      | screen         |           24 |        23 |      1 | 95.8%          |

Standard LIBERO produced 72/72 successes and is useful as an integration
control, but it is saturated for failure research. Selected LIBERO-PRO
boundary cases created natural success and failure from the same task and
initial state.

## 3. Model outputs and uncertainty scores

At query $q$, the policy receives the real agent-view image, wrist image,
proprioception and language instruction. For stochastic candidate $n$, the
main parallel mode produces an action chunk, future state and value from the
same diffusion sequence:

$$
\left(A_{q,n},\widehat I_{q+1,n},\widehat p_{q+1,n},V_{q,n}\right),
\qquad A_{q,n}\in\mathbb{R}^{16\times7}.
$$

After the selected chunk is executed, the next query uses a **new real MuJoCo
observation**, not the predicted image or proprioception.

The main candidate-level internal risks are

$$
u^{A,\mathrm{first}}_{q,n}
=\left\|
\operatorname{Std}_{k}
\widetilde A_{q,n,k,0,1:6}
\right\|_2,
\qquad
u^V_{q,n}=\operatorname{Std}_{e}\widetilde V_{q,n,e},
$$

where $k$ indexes repeated action copies in the latent frame and $e$
indexes value-latent elements. Across-candidate stochastic metrics such as
`value_std` and `value_range` are also recorded.

With candidate-wise standardization

$$
z(x_n)=
\frac{x_n-\operatorname{Mean}_m x_m}
{\operatorname{Std}_m x_m+\varepsilon},
$$

the tested combined planner was

$$
n^*=\arg\max_n
\left[
z(V_n)-\lambda
\left(w_A z(u^{A,\mathrm{first}}_n)+(1-w_A)z(u^V_n)\right)
\right].
$$

Prediction errors against reality are available only after executing the
chunk and were not used to choose that same chunk.

## 4. Failure detection

The fixed milk case used 36 calibration and 36 holdout trajectories. The
table below reports exact-query comparisons, avoiding trajectory-length
leakage.

| Query / t   | Metric                           |   Calibration AUROC |   Holdout AUROC |   Holdout AUPRC | TPR / FPR    | Alive holdout F/S   |
|:------------|:---------------------------------|--------------------:|----------------:|----------------:|:-------------|:--------------------|
| 3 / 48      | action first-step stochastic std |               0.514 |           0.616 |           0.667 | 0.10 / 0.000 | 20 / 16             |
| 3 / 48      | value stochastic std             |               0.495 |           0.653 |           0.721 | 0.10 / 0.000 | 20 / 16             |
| 3 / 48      | value stochastic range           |               0.498 |           0.656 |           0.724 | 0.10 / 0.000 | 20 / 16             |
| 8 / 128     | action latent-copy mean          |               0.944 |           0.981 |           0.987 | 0.20 / 0.000 | 20 / 13             |
| 8 / 128     | mean predicted value             |               0.996 |           0.923 |           0.868 | 0.85 / 0.077 | 20 / 13             |
| 9 / 144     | action first-step latent-copy L2 |               0.94  |           0.915 |           0.967 | 0.80 / 0.000 | 20 / 10             |
| 9 / 144     | action latent-copy mean          |               0.801 |           0.84  |           0.908 | 0.30 / 0.000 | 20 / 10             |

**Early result.** At `query=3` (`t=48`), AUROC values are weak and unstable;
there is no reliable initial-state failure detector.

**Critical-moment result.** At `query=9` (`t=144`), first-action latent-copy
inconsistency reaches holdout AUROC 0.915 and the calibration threshold gives
TPR 0.80 at FPR 0. The earliest observed physical failure occurs at `t=169`,
so the signal leads it by at least 25 simulator steps.

At `query=8`, high mean predicted value itself reaches holdout AUROC 0.923.
This is overconfidence: the model can agree on a high value and still fail.

![Detector holdout AUROC](campaigns/libero_full_validation_20260724/analysis/full_validation/plots/detector_holdout_auc.png)

The automatic sweep searched 205,920 correlated temporal variants. Perfect
rows from that sweep are exploratory and are not treated as confirmatory
evidence; the conclusions above use prespecified metrics at exact queries.

## 5. Planning: pooled confirmatory comparison

The aggregate below contains the independent milk holdout, yellow-book and
long-mug cases: 50 paired rollout seeds per formula. Milk calibration is
excluded.

| Strategy             |   Lambda | Baseline   | Strategy result   | Delta   | W/L/T    |   Exact p |
|:---------------------|---------:|:-----------|:------------------|:--------|:---------|----------:|
| action penalty       |      0.5 | 33/50      | 31/50             | -4 pp   | 6/8/36   |     0.791 |
| action penalty       |      1   | 33/50      | 28/50             | -10 pp  | 9/14/27  |     0.405 |
| action penalty       |      2   | 33/50      | 27/50             | -12 pp  | 11/17/22 |     0.345 |
| action-chunk penalty |      1   | 33/50      | 27/50             | -12 pp  | 5/11/34  |     0.21  |
| combined penalty     |      1   | 33/50      | 26/50             | -14 pp  | 6/13/31  |     0.167 |
| value penalty        |      1   | 33/50      | 26/50             | -14 pp  | 8/15/27  |     0.21  |
| value penalty        |      2   | 33/50      | 25/50             | -16 pp  | 7/15/28  |     0.134 |
| combined penalty     |      2   | 33/50      | 24/50             | -18 pp  | 8/17/25  |     0.108 |

No fixed uncertainty penalty beats `max(value)` in the pooled comparison.
The least harmful candidate is first-action penalty with $\lambda=0.5$:
31/50 versus 33/50 for the baseline. None of the paired tests is significant.

![Planning success rates](campaigns/libero_full_validation_20260724/analysis/full_validation/plots/planning_success_rates.png)

## 6. Promising but underpowered generalization cases

| Case                 | Strategy         |   Lambda | Success   | Rate   | Delta vs max(value)   |
|:---------------------|:-----------------|---------:|:----------|:-------|:----------------------|
| yellow_task8_init0   | max(value)       |      0   | 10/18     | 55.6%  | +0.0 pp               |
| yellow_task8_init0   | combined penalty |      1   | 11/18     | 61.1%  | +5.6 pp               |
| long_mug_task4_init0 | max(value)       |      0   | 9/12      | 75.0%  | +0.0 pp               |
| long_mug_task4_init0 | action penalty   |      0.5 | 11/12     | 91.7%  | +16.7 pp              |
| long_mug_task4_init0 | value penalty    |      2   | 11/12     | 91.7%  | +16.7 pp              |

Long-mug is the most positive pilot: action penalty $\lambda=0.5$ and value
penalty $\lambda=2$ each obtain 11/12 versus 9/12 for `max(value)`.
However, each comparison has only three wins and one loss
(`exact p=0.625`), so this is a replication target, not a demonstrated gain.

The action-denoising ablation is also interesting: with 10 denoising steps,
action penalty obtained 4/6 while the baseline obtained 0/6. The sample is too
small (`p=0.125`) and the baseline block was unusually difficult.

## 7. Does uncertainty predict world-model error?

Metrics and errors were rank-normalized within `split/query_idx` for
`query=0..5`.

| Uncertainty                      | Subsequent error        |   Rows |   Query-controlled rank correlation |
|:---------------------------------|:------------------------|-------:|------------------------------------:|
| action latent-copy mean          | future_wrist_mse        |    432 |                              -0.191 |
| value stochastic range           | future_wrist_mse        |    432 |                              -0.185 |
| action latent-copy mean          | future_proprio_l2       |    432 |                              -0.175 |
| action latent-copy mean          | value_abs_chunk_success |    432 |                               0.175 |
| action first-step stochastic std | future_image_mse        |    432 |                               0.153 |

All meaningful correlations satisfy $|\rho|\leq0.191$, and several have the
opposite sign. Current uncertainty scores can rank late failure risk in one
case, but they are not calibrated estimates of image or proprioception error.

## 8. Matched-seed video comparison

All twelve videos below use
`libero_spatial_with_milk/task5/init0`. Within each row, environment and
rollout seed are identical; only candidate selection changes.

These are deliberately selected pilot seeds where `max(value)` fails and at
least one alternative succeeds. They illustrate mechanism and trajectory
divergence, but **must not be used to estimate success rates**.

<h4>rollout_seed=100194</h4><table style="width:100%"><tr><td style="vertical-align:top;padding:8px;width:25%"><b>max(value)</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=220<br><video controls preload="metadata" width="250" src="final_results_media/seed_100194__max_value__fail.mp4"></video><br><a href="final_results_media/seed_100194__max_value__fail.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px;width:25%"><b>action penalty</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=170<br><video controls preload="metadata" width="250" src="final_results_media/seed_100194__action_l1__success.mp4"></video><br><a href="final_results_media/seed_100194__action_l1__success.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px;width:25%"><b>value penalty</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=220<br><video controls preload="metadata" width="250" src="final_results_media/seed_100194__value_l2__fail.mp4"></video><br><a href="final_results_media/seed_100194__value_l2__fail.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px;width:25%"><b>combined penalty</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=220<br><video controls preload="metadata" width="250" src="final_results_media/seed_100194__combined_l2__fail.mp4"></video><br><a href="final_results_media/seed_100194__combined_l2__fail.mp4">Open MP4</a></td></tr></table>
<h4>rollout_seed=100388</h4><table style="width:100%"><tr><td style="vertical-align:top;padding:8px;width:25%"><b>max(value)</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=220<br><video controls preload="metadata" width="250" src="final_results_media/seed_100388__max_value__fail.mp4"></video><br><a href="final_results_media/seed_100388__max_value__fail.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px;width:25%"><b>action penalty</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=110<br><video controls preload="metadata" width="250" src="final_results_media/seed_100388__action_l1__success.mp4"></video><br><a href="final_results_media/seed_100388__action_l1__success.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px;width:25%"><b>value penalty</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=109<br><video controls preload="metadata" width="250" src="final_results_media/seed_100388__value_l2__success.mp4"></video><br><a href="final_results_media/seed_100388__value_l2__success.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px;width:25%"><b>combined penalty</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=220<br><video controls preload="metadata" width="250" src="final_results_media/seed_100388__combined_l2__fail.mp4"></video><br><a href="final_results_media/seed_100388__combined_l2__fail.mp4">Open MP4</a></td></tr></table>
<h4>rollout_seed=100679</h4><table style="width:100%"><tr><td style="vertical-align:top;padding:8px;width:25%"><b>max(value)</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=220<br><video controls preload="metadata" width="250" src="final_results_media/seed_100679__max_value__fail.mp4"></video><br><a href="final_results_media/seed_100679__max_value__fail.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px;width:25%"><b>action penalty</b><br><span style="color:#b02a37;font-weight:700">FAIL</span> · t=220<br><video controls preload="metadata" width="250" src="final_results_media/seed_100679__action_l1__fail.mp4"></video><br><a href="final_results_media/seed_100679__action_l1__fail.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px;width:25%"><b>value penalty</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=109<br><video controls preload="metadata" width="250" src="final_results_media/seed_100679__value_l2__success.mp4"></video><br><a href="final_results_media/seed_100679__value_l2__success.mp4">Open MP4</a></td><td style="vertical-align:top;padding:8px;width:25%"><b>combined penalty</b><br><span style="color:#157347;font-weight:700">SUCCESS</span> · t=144<br><video controls preload="metadata" width="250" src="final_results_media/seed_100679__combined_l2__success.mp4"></video><br><a href="final_results_media/seed_100679__combined_l2__success.mp4">Open MP4</a></td></tr></table>

In [ ]:
# Fallback renderer for notebook frontends that hide <video> in Markdown.
import html

media = pd.read_csv(PROJECT_ROOT / "experiments/final_results_media/manifest.csv")
for seed, rows in media.groupby("rollout_seed", sort=True):
    cards = []
    for row in rows.itertuples(index=False):
        src = f"final_results_media/{row.filename}"
        outcome = "SUCCESS" if row.success else "FAIL"
        cards.append(
            '<td style="vertical-align:top;padding:8px">'
            f"<b>{html.escape(row.strategy_label)}</b><br>{outcome}, t={row.final_t}<br>"
            f'<video controls preload="metadata" width="250" src="{src}"></video>'
            f'<br><a href="{src}">Open MP4</a></td>'
        )
    display(Markdown(f"#### rollout_seed={seed}"))
    display(HTML("<table><tr>" + "".join(cards) + "</tr></table>"))

## 9. Conclusions

1. **OOD evaluation is necessary.** Standard LIBERO is saturated, while
   LIBERO-PRO provides natural mixed outcomes under fixed task/init state.
2. **There is no universal early detector yet.** Metrics before `t=80` do not
   transfer reliably from calibration to holdout.
3. **A reproducible critical-moment signal exists on one fixed case.**
   Latent action-copy inconsistency anticipates observed failure and transfers
   to a new seed block.
4. **Value overconfidence is a separate failure mode.** High mean value can be
   more predictive than value dispersion.
5. **Uncertainty is not the same as prediction error.** Image and proprioception
   error correlations remain weak after controlling episode phase.
6. **A static global penalty overfits.** The formula selected on milk
   calibration becomes substantially worse on milk holdout and does not win in
   the pooled 50-rollout aggregate.
7. **Risk handling should be conditional and phase-aware.** Penalizing every
   query with one global $\lambda$ is too crude.
8. **LIBERO-Safety remains an unexecuted validation axis.** PRO drop and
   wrong-object heuristics do not replace official safety constraints.

## 10. Next testable planner

Keep `max(value)` as the default and enable risk-aware behavior only after a
calibrated task/query-specific trigger:

$$
g_q=\mathbb{1}\left[u_q>\tau_{\alpha,\mathrm{task},q}\right],
$$

$$
n_q^*=\arg\max_n
\left[z(V_{q,n})-g_q\lambda_q z(u^{A,\mathrm{first}}_{q,n})\right].
$$

When $g_q=1$, also increase candidates $N:4\rightarrow8$, shorten the
executed horizon $H:16\rightarrow4$ or $8$, and replan from the next real
observation. Calibration must be frozen on milk and evaluated unchanged on
milk holdout, yellow, long-mug and official LIBERO-Safety suites.

## 11. Reproducibility

- [Complete report](LIBERO_COMPLETE_RESULTS_20260724.md)
- [Frozen 8-hour protocol](LIBERO_8H_VALIDATION_PROTOCOL.md)
- [Machine-readable full-validation tables](campaigns/libero_full_validation_20260724/analysis/full_validation/)
- [Experiment grid](configs/libero_campaign_8h.json)
- [Video manifest](final_results_media/manifest.csv)
- [Paper and benchmark review](../articles/LIBERO_EXPERIMENTS_AND_PAPERS.md)

Regenerate this notebook:

```bash
/home/alexander/venvs/cosmos_policy_libero/bin/python   scripts/build_final_results_notebook.py
```